In [1]:
import uproot
import matplotlib.pyplot as plt
import seaborn
import numpy as np
import math
from scipy.fft import fft, fftfreq
import plotly.express as px
import plotly.io as pio
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import os
from plotly.subplots import make_subplots
from plotly import tools
import plotly.offline as pyo
import sys

In [9]:
Run_num = '12049'
#files =uproot.open(f"/Users/danielcarber/Documents/SBND/Noise Analysis/noise_output_run{Run_num}.root")
files =uproot.open(f"/Users/danielcarber/Documents/SBND/Noise Analysis/data/noise_output_coh_run{Run_num}.root")
files['tpc_noise;1'].keys()

['raw_rms', 'coh_wave']

In [10]:
raw_rms = files['tpc_noise;1']['raw_rms'].array().to_list()
Noise_df = {'Channel_id':[],'Raw_rms':[],'wire_plane':[]}
for r,rms in enumerate(raw_rms):
    Noise_df['Channel_id'].append(r)
    Noise_df['Raw_rms'].append(rms)
    if r <1984:
        Noise_df['wire_plane'].append('UB')
    elif r<3968:
        Noise_df['wire_plane'].append('VB')
    elif r<5632:
        Noise_df['wire_plane'].append('YB')
    elif r<7616:
        Noise_df['wire_plane'].append('UA')
    elif r<9600:
        Noise_df['wire_plane'].append('VA')
    else:
        Noise_df['wire_plane'].append('YA')
Noise_df = pd.DataFrame(Noise_df)

In [11]:
Argon_df = pd.read_excel("/Users/danielcarber/Documents/SBND/Noise Analysis/sbn_noise_repo/sbnd/datafiles/SBND_LAr_level.xlsx")
print(Argon_df['Lar_level'][27])
Argon_df['Lar_level'] = Argon_df['Lar_level']/Argon_df['Lar_level'][28]*100
#print(datetime.fromtimestamp(Argon_df['Time'][0]))

516.42


In [12]:
wire_plane_list = ['UB','VB','YB','UA','VA','YA']
wire_df = {'Channel_id':[],'cryo':[],'tpc':[],'tpc':[],'plane':[],'rel_wire':[],'x_0':[],'y_0':[],'z_0':[],'x_1':[],'y_1':[],'z_1':[],'r':[]}
wire_txt = '/Users/danielcarber/Documents/SBND/Noise Analysis/sbn_noise_repo/sbnd/datafiles/Wire_lengths.txt'

with open(wire_txt) as f:
    for line in f:
        #print(line)
        currentline = line.split(" ")
        #print(currentline)
        wire_df['Channel_id'].append(int(currentline[0]))
        wire_df['cryo'].append(int(currentline[1]))
        wire_df['tpc'].append(int(currentline[2]))
        wire_df['plane'].append(int(currentline[3]))
        wire_df['rel_wire'].append(int(currentline[4]))
        wire_df['x_0'].append(float(currentline[5]))
        wire_df['y_0'].append(float(currentline[6]))
        wire_df['z_0'].append(float(currentline[7]))
        wire_df['x_1'].append(float(currentline[8]))
        wire_df['y_1'].append(float(currentline[9]))
        z_1 = currentline[10]
        #print(z_1[:-2])
        wire_df['z_1'].append(float(currentline[10][:-2]))
        length = np.sqrt(np.square(float(currentline[8])-float(currentline[5]))+np.square(float(currentline[9])-float(currentline[6]))+np.square(float(currentline[10][:-2])-float(currentline[7])))
        wire_df['r'].append(length)
wire_df = pd.DataFrame(wire_df)        
print(wire_df['r'])
Noise_df =Noise_df.merge(wire_df, on='Channel_id')
print(Noise_df)

0          1.039423
1          1.732038
2          2.424653
3          3.117769
4          3.810384
            ...    
11259    400.000000
11260    400.000000
11261    400.000000
11262    400.000000
11263    400.000000
Name: r, Length: 11264, dtype: float64
       Channel_id   Raw_rms wire_plane  cryo  tpc  plane  rel_wire     x_0  \
0               0  0.507885         UB     0    0      0         0 -201.45   
1               1  0.507885         UB     0    0      0         1 -201.45   
2               2  0.507885         UB     0    0      0         2 -201.45   
3               3  0.507885         UB     0    0      0         3 -201.45   
4               4  0.507885         UB     0    0      0         4 -201.45   
...           ...       ...        ...   ...  ...    ...       ...     ...   
11259       11259  0.920608         YA     0    1      2      1659  202.05   
11260       11260  0.920608         YA     0    1      2      1660  202.05   
11261       11261  0.920608         YA 

In [13]:
def Extract(lst,pos):
    return [item[pos] for item in lst]
print(Times)

NameError: name 'Times' is not defined

In [14]:
a, b = np.polyfit(Noise_df['r'],Noise_df['Raw_rms'], 1)
correct_rms = []
print(a)
for i in range(0,len(Noise_df['r']),32):
    correct_rms.extend(a*Noise_df['r'][i:i+32] +Noise_df['Raw_rms'][i]-a*Noise_df['r'][i+16])
Noise_df['corrected_rms'] = correct_rms
fig = make_subplots(rows=1,cols=1)

fig.add_trace(go.Scatter(x=Noise_df['r'],y = Noise_df['Raw_rms'],marker_color = 'green',mode='markers'),row = 1, col = 1)

fig.add_trace(go.Scatter(x=Noise_df['r'],y = Noise_df['corrected_rms'],marker_color = 'red',mode='markers',opacity=.7),row = 1, col = 1)
fig.add_trace(go.Scatter(x=Noise_df['r'],y = a*Noise_df['r']+b,marker_color = 'blue'),row = 1, col = 1)

#fig.update_layout(xaxis2 = dict(range = [0,median+5]))
#fig.update_layout(margin = dict(r=200))
#fig.add_annotation(dict(font = dict(size = 10),xshift= 180,yshift=120,text = f"Mean RMS:{mean:.2f}",showarrow = False),row =1,col=2)
#fig.add_annotation(dict(font = dict(size = 10),xshift= 180,yshift=110,text = f"Median RMS:{median:.2f}",showarrow = False),row =1,col=2)


fig.update_yaxes(title_text = "RMS [ADC]",row = 1, col = 1)

fig.update_xaxes(title_text = "Wire length [cm]",row = 1, col = 1)

#fig.update_layout(xaxis2 = dict(range = [0,5]),xaxis4 = dict(range = [0,5]),xaxis6 = dict(range = [0,5]))
#fig.update_layout(yaxis = dict(range = [0,5]),yaxis3 = dict(range = [0,5]),yaxis5 = dict(range = [0,5]))
fig.update_layout(xaxis = dict(range= [-10,700],tickmode = 'linear',dtick = 100))
fig.update_layout(height = 800, width = 1200,showlegend = False)

fig.write_image(f'/Users/danielcarber/Documents/SBND/Noise Analysis/Plots/RMS_vs_wire_length.png')

fig.show()

0.001249039593635577


In [8]:
#filename = f"RMS_plots_planeB_{Run_num}.pdf"
dt = timedelta(hours=12)
fig = make_subplots(rows=3,cols=2,subplot_titles = (f'UB Mean RMS vs Time','UA Mean RMS vs Time',f'VB Mean RMS vs Time','VA Mean RMS vs Time',f'YB Mean RMS vs Time','YA Mean RMS vs Time',),
                                                   specs=[[{"secondary_y": True}, {"secondary_y": True}],
                           [{"secondary_y": True}, {"secondary_y": True}], [{"secondary_y": True}, {"secondary_y": True}]])
#mask = Noise_df['wire_plane'] == 'UB'
#median = np.median(Noise_df['Raw_rms'][mask])
#mean = np.mean(Noise_df['Raw_rms'][mask])
#fig.add_trace(go.Histogram(x=Noise_df['Raw_rms'][mask],marker_color = 'red',xbins=dict(start = median - 5,end = median+5,size=.05)),row = 1, col =2)
'''
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],0),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 1, col = 1)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],1),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 2, col = 1)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],2),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 3, col = 1)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],3),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 1, col = 2)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],4),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 2, col = 2)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],5),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 3, col = 2)
'''
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_mean'],0),marker_color = 'red',mode='lines+markers',name = 'RMS'),row = 1, col = 1,secondary_y = False)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_mean'],1),marker_color = 'red',mode='lines+markers',showlegend=False),row = 2, col = 1,secondary_y = False)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_mean'],2),marker_color = 'red',mode='lines+markers',showlegend=False),row = 3, col = 1,secondary_y = False)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_mean'],3),marker_color = 'red',mode='lines+markers',showlegend=False),row = 1, col = 2,secondary_y = False)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_mean'],4),marker_color = 'red',mode='lines+markers',showlegend=False),row = 2, col = 2,secondary_y = False)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_mean'],5),marker_color = 'red',mode='lines+markers',showlegend=False),row = 3, col = 2,secondary_y = False)

fig.add_trace(go.Scatter(x=Times,y = Temp_df['AT'],marker_color = 'blue',mode='lines+markers',name = 'Temp A Plane Top',opacity=0.5),row = 1, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['AT'],marker_color = 'blue',mode='lines+markers',showlegend=False,opacity=0.5),row = 2, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['AT'],marker_color = 'blue',mode='lines+markers',showlegend=False,opacity=0.5),row = 3, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['AT'],marker_color = 'blue',mode='lines+markers',showlegend=False,opacity=0.5),row = 1, col = 2,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['AT'],marker_color = 'blue',mode='lines+markers',showlegend=False,opacity=0.5),row = 2, col = 2,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['AT'],marker_color = 'blue',mode='lines+markers',showlegend=False,opacity=0.5),row = 3, col = 2,secondary_y = True)

fig.add_trace(go.Scatter(x=Times,y = Temp_df['AB'],marker_color = 'cyan',mode='lines+markers',name = 'Temp A Plane Bottom',opacity=0.5),row = 1, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['AB'],marker_color = 'cyan',mode='lines+markers',showlegend=False,opacity=0.5),row = 2, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['AB'],marker_color = 'cyan',mode='lines+markers',showlegend=False,opacity=0.5),row = 3, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['AB'],marker_color = 'cyan',mode='lines+markers',showlegend=False,opacity=0.5),row = 1, col = 2,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['AB'],marker_color = 'cyan',mode='lines+markers',showlegend=False,opacity=0.5),row = 2, col = 2,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['AB'],marker_color = 'cyan',mode='lines+markers',showlegend=False,opacity=0.5),row = 3, col = 2,secondary_y = True)

fig.add_trace(go.Scatter(x=Times,y = Temp_df['BT'],marker_color = 'green',mode='lines+markers',name = 'Temp B. Plane Top',opacity=0.5),row = 1, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['BT'],marker_color = 'green',mode='lines+markers',showlegend=False,opacity=0.5),row = 2, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['BT'],marker_color = 'green',mode='lines+markers',showlegend=False,opacity=0.5),row = 3, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['BT'],marker_color = 'green',mode='lines+markers',showlegend=False,opacity=0.5),row = 1, col = 2,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['BT'],marker_color = 'green',mode='lines+markers',showlegend=False,opacity=0.5),row = 2, col = 2,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['BT'],marker_color = 'green',mode='lines+markers',showlegend=False,opacity=0.5),row = 3, col = 2,secondary_y = True)

fig.add_trace(go.Scatter(x=Times,y = Temp_df['BB'],marker_color = 'lime',mode='lines+markers',name = 'Temp B Plane Bottom',opacity=0.5),row = 1, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['BB'],marker_color = 'lime',mode='lines+markers',showlegend=False,opacity=0.5),row = 2, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['BB'],marker_color = 'lime',mode='lines+markers',showlegend=False,opacity=0.5),row = 3, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['BB'],marker_color = 'lime',mode='lines+markers',showlegend=False,opacity=0.5),row = 1, col = 2,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['BB'],marker_color = 'lime',mode='lines+markers',showlegend=False,opacity=0.5),row = 2, col = 2,secondary_y = True)
fig.add_trace(go.Scatter(x=Times,y = Temp_df['BB'],marker_color = 'lime',mode='lines+markers',showlegend=False,opacity=0.5),row = 3, col = 2,secondary_y = True)
def improve_text_position(x):
    """ it is more efficient if the x values are sorted """
    # fix indentation 
    positions = ['top center', 'bottom center']  # you can add more: left center ...
    return [positions[i % len(positions)] for i in range(len(x))]

fig.update_traces(textposition=improve_text_position(Extract(run_df['run_mean'],0)),textfont_size=12)

fig.update_layout(yaxis = dict(range = [1,3.5]),xaxis = dict(range = [min(Times)-dt,max(Times)+dt]))
fig.update_layout(yaxis3 = dict(range = [1,3.5]),xaxis2 = dict(range = [min(Times)-dt,max(Times)+dt]))
fig.update_layout(yaxis5 = dict(range = [1,3.5]),xaxis3 = dict(range = [min(Times)-dt,max(Times)+dt]))
fig.update_layout(yaxis7 = dict(range = [1,3.5]),xaxis4 = dict(range = [min(Times)-dt,max(Times)+dt]))
fig.update_layout(yaxis9 = dict(range = [1,3.5]),xaxis5 = dict(range = [min(Times)-dt,max(Times)+dt]))
fig.update_layout(yaxis11 = dict(range = [1,3.5]),xaxis6 = dict(range = [min(Times)-dt,max(Times)+dt]))



fig.update_yaxes(title_text = "Mean RMS [ADC]", secondary_y=False,row = 1, col = 1)
fig.update_yaxes(title_text = "Mean RMS [ADC]", secondary_y=False,row = 2, col = 1)
fig.update_yaxes(title_text = "Mean RMS [ADC]", secondary_y=False,row = 3, col = 1)
fig.update_yaxes(title_text = "Temp [K]", secondary_y=True,row = 1, col = 1)
fig.update_yaxes(title_text = "Temp [K]", secondary_y=True,row = 2, col = 1)
fig.update_yaxes(title_text = "Temp [K]", secondary_y=True,row = 3, col = 1)
fig.update_xaxes(title_text = "Time",row = 1, col = 1)
fig.update_xaxes(title_text = "Time",row = 2, col = 1)
fig.update_xaxes(title_text = "Time",row = 3, col = 1)
fig.update_xaxes(title_text = "Time",row = 1, col = 2)
fig.update_xaxes(title_text = "Time",row = 2, col = 2)
fig.update_xaxes(title_text = "Time",row = 3, col = 2)
fig.update_yaxes(title_text = "Mean RMS [ADC]", secondary_y=False,row = 1, col = 2)
fig.update_yaxes(title_text = "Mean RMS [ADC]", secondary_y=False,row = 2, col = 2)
fig.update_yaxes(title_text = "Mean RMS [ADC]", secondary_y=False,row = 3, col = 2)
fig.update_yaxes(title_text = "Temp [K]", secondary_y=True,row = 1, col = 2)
fig.update_yaxes(title_text = "Temp [K]", secondary_y=True,row = 2, col = 2)
fig.update_yaxes(title_text = "Temp [K]", secondary_y=True,row = 3, col = 2)


fig.update_layout(height = 1200, width = 1600,showlegend = True)
first_run = run_df['run'][0]
last_run = run_df['run'][-1]
fig.write_image(f'/Users/danielcarber/Documents/SBND/Noise Analysis/Plots/RMS_vs_wire_length_{first_run}.png')
#fig.write_image(directory+filename)
fig.show()

NameError: name 'Times' is not defined

In [ ]:
#filename = f"RMS_plots_planeB_{Run_num}.pdf"
dt = timedelta(hours=12)
fig = make_subplots(rows=1,cols=1,subplot_titles = ('TPC Noise, LAr level, and Temperature Vs. Time',),
                                                   specs=[[{"secondary_y": True}]])
#mask = Noise_df['wire_plane'] == 'UB'
#median = np.median(Noise_df['Raw_rms'][mask])
#mean = np.mean(Noise_df['Raw_rms'][mask])
#fig.add_trace(go.Histogram(x=Noise_df['Raw_rms'][mask],marker_color = 'red',xbins=dict(start = median - 5,end = median+5,size=.05)),row = 1, col =2)
'''
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],0),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 1, col = 1)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],1),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 2, col = 1)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],2),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 3, col = 1)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],3),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 1, col = 2)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],4),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 2, col = 2)
fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_median'],5),marker_color = 'red',text = run_df['run'],mode='lines+markers+text'),row = 3, col = 2)
'''

fig.add_trace(go.Scatter(x=Times,y = Extract(run_df['run_mean'],6),marker_color = 'red',mode='lines+markers',showlegend=True,name = 'Noise'),row = 1, col = 1,secondary_y = False)

fig.add_trace(go.Scatter(x=Times,y = Temp_df['mean'],marker_color = 'blue',mode='lines+markers',name = 'Average Temp',opacity=0.5),row = 1, col = 1,secondary_y = True)
fig.add_trace(go.Scatter(x=Argon_df['Time'],y = Argon_df['Lar_level'],marker_color = 'green',mode='lines+markers',name = 'Argon Level',opacity=0.5),row = 1, col = 1,secondary_y = True)

def improve_text_position(x):
    """ it is more efficient if the x values are sorted """
    # fix indentation 
    positions = ['top center', 'bottom center']  # you can add more: left center ...
    return [positions[i % len(positions)] for i in range(len(x))]

fig.update_traces(textposition=improve_text_position(Extract(run_df['run_median'],0)),textfont_size=12)

fig.update_layout(yaxis = dict(range = [1,3.5]),xaxis = dict(range = [min(Times)-dt,max(Times)+dt]))



fig.update_yaxes(title_text = "Median RMS [ADC]", secondary_y=False,row = 1, col = 1,titlefont = dict(size=20),tickfont = dict(size=20))

fig.update_yaxes(title_text = "Temp [K] and LAr Level %", secondary_y=True,row = 1, col = 1,titlefont = dict(size=20),tickfont = dict(size=20))

fig.update_xaxes(title_text = "Time",row = 1, col = 1,titlefont = dict(size=20),tickfont = dict(size=20))


fig.update_annotations(font_size=30)


fig.update_layout(height = 600, width = 1200,showlegend = True,font = dict(size=20),legend=dict(
    yanchor="top",
    y=0.99,
    xanchor="right",
    x=0.90,
    bgcolor="LightSteelBlue",
        bordercolor="Black",
))
first_run = run_df['run'][0]
last_run = run_df['run'][-1]
fig.write_image(f'/Users/danielcarber/Documents/SBND/Noise Analysis/Plots/Mean_RMS+temp_vs_{first_run}_{last_run}.png')
#fig.write_image(directory+filename)
fig.show()